# Day 3 - Conversational AI - aka Chatbot!

In [1]:
# Import required libraries
import gradio as gr # oh yeah!
from IPython.display import Markdown, display

In [2]:
%run ../config/llm_settings.py

In [3]:
%run ../utils/llm_functions.py

In [4]:
validaciones = validate_loaded_clients()

validaciones

{'ollama': {'provider': 'ollama',
  'model': 'gemma4:12b',
  'client_loaded': True,
  'model_configured': True,
  'connection_test': 'SKIPPED',
  'status': 'PASSED',
  'error': None},
 'nvidia': {'provider': 'nvidia',
  'model': 'z-ai/glm-5.2',
  'client_loaded': True,
  'model_configured': True,
  'connection_test': 'SKIPPED',
  'status': 'PASSED',
  'error': None},
 'openai': {'provider': 'openai',
  'model': 'gpt-4o-mini',
  'client_loaded': True,
  'model_configured': True,
  'connection_test': 'SKIPPED',
  'status': 'PASSED',
  'error': None},
 'anthropic': {'provider': 'anthropic',
  'model': 'claude-opus-4-1-20250805',
  'client_loaded': True,
  'model_configured': True,
  'connection_test': 'SKIPPED',
  'status': 'PASSED',
  'error': None},
 'google': {'provider': 'google',
  'model': 'gemini-2.5-flash-lite',
  'client_loaded': True,
  'model_configured': True,
  'connection_test': 'SKIPPED',
  'status': 'PASSED',
  'error': None}}

In [10]:
# Again, I'll be in scientist-mode and change this global during the lab

system_message = "You are a helpful assistant".strip()

## And now, writing a new callback

We now need to write a function called:

`chat(message, history)`

Which will be a callback function we will give gradio.

### The job of this function

Take a message, take the prior conversation, and return the response.


In [11]:
def chat(message, history):
    message = (message or "").strip()

    if not message:
        yield "Escribe un mensaje."
        return

    try:
        # Versión de aprendizaje:
        # convertimos el historial a texto y lo añadimos al prompt.
        history_text = ""

        for item in history or []:
            role = item.get("role", "")
            content = item.get("content", "")

            if isinstance(content, str) and content.strip():
                history_text += (
                    f"{role.upper()}: {content.strip()}\n"
                )

        user_prompt = f"""
Historial de la conversación:
{history_text or "Sin mensajes anteriores."}

Nuevo mensaje del usuario:
{message}
""".strip()

        yield from stream_provider(
            provider="ollama",
            system_msg=system_message,
            user_msg=user_prompt,
            max_tokens=1000,
            temperature=0.7,
        )

    except Exception as error:
        yield (
            "Error al consultar Ollama: "
            f"{type(error).__name__}: {error}"
        )

In [12]:
gr.close_all()

demo = gr.ChatInterface(
    fn=chat, title="Chat con Ollama", description="Chat con historial y respuesta en streaming.",)
demo.queue(default_concurrency_limit=1,)
demo.launch(share=True,)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://4303fa549a7f48cd63.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## OK let's keep going!

Using a system message to add context, and to give an example answer.. this is "one shot prompting" again

In [13]:
system_message = "Eres un asistente útil en una tienda de ropa. \
Debes tratar de alentar gentilmente al cliente a que pruebe los artículos que están en oferta.\
Los sombreros tienen un 60 % de descuento y la mayoría de los demás artículos tienen un 50 % de descuento. \
Por ejemplo, si el cliente dice 'Quiero comprar un sombrero', \
podrías responder algo como 'Genial, tenemos muchos sombreros, incluidos varios que son parte de nuestro evento de rebajas'. \
Anima al cliente a comprar sombreros si no está seguro de qué comprar."

In [14]:
gr.close_all()

demo = gr.ChatInterface(
    fn=chat, title="Chat con Ollama", description="Chat con historial y respuesta en streaming.",)
demo.queue(default_concurrency_limit=1,)
demo.launch(share=True,)

Closing server running on port: 7861
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://df87c7227dab690a2b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [15]:
system_message += "\nSi el cliente pide zapatos, debes responder que los zapatos no están en oferta hoy, \
¡pero recuérdale al cliente que mire los sombreros!"

In [16]:
gr.close_all()

demo = gr.ChatInterface(
    fn=chat, title="Chat con Ollama", description="Chat con historial y respuesta en streaming.",)
demo.queue(default_concurrency_limit=1,)
demo.launch(share=True,)

Closing server running on port: 7861
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://1723688b8448e8d758.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [20]:
import unicodedata

def normalize_text(text: str) -> str:
    text = (text or "").lower().strip()

    return "".join(
        character
        for character in unicodedata.normalize("NFD", text)
        if unicodedata.category(character) != "Mn"
    )

In [21]:
def chat(message, history):
    message = (message or "").strip()

    if not message:
        yield "Escribe un mensaje."
        return

    normalized_message = normalize_text(message)
    contextual_system_message = system_message

    if (
        "cinturon" in normalized_message
        or "cinturones" in normalized_message
    ):
        contextual_system_message += """
Regla comercial importante:
La tienda no vende cinturones. Indícalo claramente al usuario y
recomienda otros artículos disponibles o en oferta. No inventes
productos, precios ni promociones que no estén en el contexto.
""".rstrip()

    history_lines = []

    for item in history or []:
        role = item.get("role", "")
        content = item.get("content", "")

        if isinstance(content, str) and content.strip():
            history_lines.append(
                f"{role.upper()}: {content.strip()}"
            )

    history_text = "\n".join(history_lines)

    prompt = f"""
Historial de conversación:
{history_text or "Sin historial anterior."}

Nuevo mensaje:
{message}
""".strip()

    try:
        yield from stream_provider(
            provider="ollama",
            system_msg=contextual_system_message,
            user_msg=prompt,
            max_tokens=1000,
            temperature=0.7,
        )

    except Exception as error:
        yield (
            f"Error al consultar el modelo: "
            f"{type(error).__name__}: {error}"
        )

In [22]:
gr.close_all()

demo = gr.ChatInterface(
    fn=chat, title="Chat con Ollama", description="Chat con historial y respuesta en streaming.",)
demo.queue(default_concurrency_limit=1,)
demo.launch(share=True,)

Closing server running on port: 7861
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://43eadcf982a9a131f0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
